# Open-Source LLMs on Groq — Demo Notebook

Use **free open-source models** via [Groq](https://groq.com) — the fastest LLM inference engine.

> **Prerequisites:** Get a free `GROQ_API_KEY` at https://console.groq.com/keys and add it to your `.env` file.

```
# Uncomment and run to install all dependencies:
# !pip install groq python-dotenv ddgs
```

## 1. Setup

In [71]:
import os, json, time
from dotenv import load_dotenv
from groq import Groq

load_dotenv()
api_key = os.environ.get("GROQ_API_KEY", "")
if not api_key:
    raise ValueError("GROQ_API_KEY not found in .env — get one free at https://console.groq.com/keys")
if api_key.startswith("xai-"):
    raise ValueError("That's an xAI (Grok) key, not a Groq key. Groq keys start with 'gsk_'.")

client = Groq(api_key=api_key)
print(f"Groq client ready (key: gsk_...{api_key[-4:]})")

Groq client ready (key: gsk_...CELr)


## 2. Available Models

All models below are **free** on Groq's developer tier.

| Model ID | Params | Speed | Best For |
|---|---|---|---|
| `llama-3.1-8b-instant` | 8B | ~560 tok/s | Fast drafts, classification, batch jobs |
| `llama-3.3-70b-versatile` | 70B | ~280 tok/s | Complex reasoning, reports — **best all-rounder** |
| `openai/gpt-oss-120b` | 120B | ~500 tok/s | Highest quality open model |
| `openai/gpt-oss-20b` | 20B | ~1000 tok/s | Fastest large model |
| `meta-llama/llama-4-scout-17b-16e-instruct` | 17B MoE | ~750 tok/s | Multimodal (images + text), latest Llama 4 |
| `qwen/qwen3-32b` | 32B | ~400 tok/s | Multilingual (Hindi!), math, reasoning |
| `moonshotai/kimi-k2-instruct` | Large | — | Strong on coding and reasoning |

**Change the variable below to experiment with any model!**

In [72]:
# ╔════════════════════════════════════════════════════════╗
# ║  CHANGE THIS to try different models                  ║
# ╚════════════════════════════════════════════════════════╝
SELECTED_MODEL = "llama-3.3-70b-versatile"

# Fetch live model list from Groq
models = client.models.list()
print(f"Selected: {SELECTED_MODEL}\n")
print(f"{'Model ID':<50} {'Owner':<20} {'Context'}")
print("─" * 80)
for m in sorted(models.data, key=lambda x: x.id):
    ctx = getattr(m, 'context_window', 'N/A')
    marker = " ◄" if m.id == SELECTED_MODEL else ""
    print(f"{m.id:<50} {m.owned_by:<20} {ctx}{marker}")

Selected: llama-3.3-70b-versatile

Model ID                                           Owner                Context
────────────────────────────────────────────────────────────────────────────────
allam-2-7b                                         SDAIA                4096
canopylabs/orpheus-arabic-saudi                    Canopy Labs          4000
canopylabs/orpheus-v1-english                      Canopy Labs          4000
groq/compound                                      Groq                 131072
groq/compound-mini                                 Groq                 131072
llama-3.1-8b-instant                               Meta                 131072
llama-3.3-70b-versatile                            Meta                 131072 ◄
meta-llama/llama-4-scout-17b-16e-instruct          Meta                 131072
meta-llama/llama-prompt-guard-2-22m                Meta                 512
meta-llama/llama-prompt-guard-2-86m                Meta                 512
moonshotai/kimi-k2-instr

---
## 3. Basic Chat Completion

In [73]:
response = client.chat.completions.create(
    model=SELECTED_MODEL,
    messages=[
        {"role": "system", "content": "You are a helpful assistant. Be concise."},
        {"role": "user", "content": "What are the top 3 metrics to evaluate a tech company's financial health?"}
    ],
    temperature=0.5, max_tokens=300,
)
print(response.choices[0].message.content)
print(f"\n[{response.usage.prompt_tokens} in / {response.usage.completion_tokens} out | {response.model}]")

The top 3 metrics to evaluate a tech company's financial health are:

1. **Revenue Growth Rate**: Measures the increase in revenue over time.
2. **Gross Margin**: Indicates profitability by calculating the difference between revenue and cost of goods sold.
3. **Cash Flow**: Shows a company's ability to generate cash and pay debts, often measured by Free Cash Flow (FCF) or Operating Cash Flow.

[60 in / 85 out | llama-3.3-70b-versatile]


---
## 4. Stock Sentiment Analysis

In [74]:
headlines = [
    "NVIDIA beats Q4 earnings expectations, data center revenue up 93%",
    "NVIDIA announces $50B stock buyback program",
    "Analysts warn NVIDIA's AI growth may slow as competition heats up",
    "China export restrictions could impact NVIDIA's 2026 revenue by 15%",
    "NVIDIA partners with major automakers for autonomous driving chips",
]

prompt = f"""Analyze sentiment of these NVIDIA headlines.
For each: Sentiment (Bullish/Bearish/Neutral), Confidence, brief reason.
End with an overall score from -1.0 (bearish) to +1.0 (bullish).

Headlines:
{chr(10).join(f'{i+1}. {h}' for i, h in enumerate(headlines))}"""

r = client.chat.completions.create(
    model=SELECTED_MODEL,
    messages=[{"role": "system", "content": "Expert financial sentiment analyst."}, {"role": "user", "content": prompt}],
    temperature=0.3, max_tokens=800,
)
print(r.choices[0].message.content)

Here's the analysis of each headline:

1. **NVIDIA beats Q4 earnings expectations, data center revenue up 93%**
   - Sentiment: Bullish
   - Confidence: High
   - Reason: Beating earnings expectations and significant revenue growth in a key segment is a strong positive indicator.

2. **NVIDIA announces $50B stock buyback program**
   - Sentiment: Bullish
   - Confidence: High
   - Reason: A large stock buyback program indicates confidence in the company's financials and a commitment to returning value to shareholders.

3. **Analysts warn NVIDIA's AI growth may slow as competition heats up**
   - Sentiment: Bearish
   - Confidence: Medium
   - Reason: Potential slowing of growth in a key area due to increasing competition could negatively impact the company's future performance.

4. **China export restrictions could impact NVIDIA's 2026 revenue by 15%**
   - Sentiment: Bearish
   - Confidence: Medium
   - Reason: Significant potential revenue impact from export restrictions poses a nota

---
## 5. Investment Thesis — Reliance Industries

In [75]:
reliance = """
Company: Reliance Industries Ltd (RELIANCE.NS)
Market Cap: ₹20.1 Lakh Crore (~$240B) | P/E: 28 | P/B: 2.8
Revenue: ₹10.1 Lakh Crore (+7.5% YoY) | Net Margin: 8.2% | ROE: 9.1%
Segments: O2C (Petrochemicals), Jio (490M+ subscribers), Retail (18K+ stores), New Energy
Debt/Equity: 0.36 | Promoter Holding: 50.3%
Recent: Jio AI Cloud launch, solar & green hydrogen investments
"""

r = client.chat.completions.create(
    model=SELECTED_MODEL,
    messages=[
        {"role": "system", "content": "Senior Indian equity analyst at a top brokerage."},
        {"role": "user", "content": f"Generate a brief investment thesis for Reliance. Include Rating, Bull Case, Bear Case, Key Catalyst, 1-year target in INR.\n{reliance}"}
    ],
    temperature=0.4, max_tokens=800,
)
print(r.choices[0].message.content)

**Investment Thesis:**
We initiate coverage on Reliance Industries Ltd (RELIANCE.NS) with a **BUY** rating.

**Bull Case:** Reliance's diversified business portfolio, driven by the growth of Jio's subscriber base and increasing profitability in the O2C segment, will lead to a significant increase in earnings. The company's foray into New Energy, including solar and green hydrogen, is expected to drive long-term growth and provide a competitive edge. We anticipate Jio's AI Cloud launch to further augment its digital services offerings, driving revenue growth.

**Bear Case:** Intensifying competition in the telecom sector, potential delays in the ramp-up of New Energy initiatives, and volatility in global energy markets may impact Reliance's profitability. Additionally, high capex requirements for Jio's 5G rollout and New Energy investments may lead to increased debt levels, affecting the company's return on equity.

**Key Catalyst:** Successful execution of Jio's 5G rollout, coupled wit

---
## 6. Model Comparison — Same Prompt, 6 Models

See how speed and quality differ across models.

In [76]:
MODELS = [
    "llama-3.1-8b-instant",
    "llama-3.3-70b-versatile",
    "openai/gpt-oss-20b",
    "openai/gpt-oss-120b",
    "qwen/qwen3-32b",
    "meta-llama/llama-4-scout-17b-16e-instruct",
]

prompt = """Tesla: P/E=170, Revenue Growth=+25%, Gross Margin=18.2%, FCF=$3.6B, Beta=2.05.
In exactly 3 bullet points: Is Tesla overvalued?"""

print("MODEL COMPARISON: Is Tesla Overvalued?")
print("=" * 85)

for mid in MODELS:
    try:
        t0 = time.time()
        r = client.chat.completions.create(
            model=mid,
            messages=[{"role": "system", "content": "Concise equity analyst."}, {"role": "user", "content": prompt}],
            temperature=0.3, max_tokens=300,
        )
        dt = time.time() - t0
        tok = r.usage.completion_tokens
        print(f"\n─── {mid}  |  {dt:.2f}s  |  {tok} tok  |  {tok/dt:.0f} tok/s ───")
        print(r.choices[0].message.content)
    except Exception as e:
        print(f"\n[{mid}] Error: {e}")
    time.sleep(1)

MODEL COMPARISON: Is Tesla Overvalued?

─── llama-3.1-8b-instant  |  0.71s  |  249 tok  |  350 tok/s ───
Based on the provided information, here are three points to consider whether Tesla is overvalued:

* **High P/E ratio**: With a P/E ratio of 170, Tesla's stock price is significantly higher than its earnings, which may indicate that the stock is overvalued. A P/E ratio above 30 is generally considered high, and 170 is extremely high, suggesting that investors are expecting high growth rates or are speculating on the company's future prospects.

* **Revenue growth and gross margin**: While Tesla's revenue growth of 25% is impressive, its gross margin of 18.2% is relatively low compared to other companies in the same industry. This may indicate that Tesla is still in the process of scaling its operations and optimizing its manufacturing processes, which could impact its profitability and make the stock more vulnerable to market fluctuations.

* **Beta and valuation**: Tesla's beta of 

---
## 7. Streaming + JSON Mode

In [77]:
# 7a. Streaming — see tokens arrive in real-time
stream = client.chat.completions.create(
    model=SELECTED_MODEL,
    messages=[{"role": "user", "content": "Compare growth vs value investing in 2026. Keep to 100 words."}],
    temperature=0.5, max_tokens=300, stream=True,
)
print("Streaming: ", end="")
for chunk in stream:
    if chunk.choices[0].delta.content:
        print(chunk.choices[0].delta.content, end="", flush=True)
print("\n")

Streaming: In 2026, growth investing focuses on companies with high potential for expansion, often in tech and healthcare. Value investing targets undervalued companies with strong fundamentals. Growth investing offers higher returns but with higher risk, while value investing provides more stability with lower returns. As interest rates rise, value investing may gain appeal, while growth investing may slow due to increased borrowing costs. Ultimately, a balanced portfolio combining both strategies can optimize returns and manage risk. Diversification is key to navigating the 2026 market.



In [78]:
# 7b. JSON Mode — structured extraction
r = client.chat.completions.create(
    model=SELECTED_MODEL,
    messages=[
        {"role": "system", "content": "Extract structured data. Respond in valid JSON only."},
        {"role": "user", "content": """Extract: \"Amazon Q4 revenue $187.8B (+10% YoY). AWS $24.2B (+19%). Operating income $13.2B. Guided Q1 $155-160B.\"
Keys: company, ticker, quarter, revenue_b, revenue_growth_pct, aws_revenue_b, operating_income_b, guidance_low_b, guidance_high_b"""}
    ],
    temperature=0.0, max_tokens=300, response_format={"type": "json_object"},
)
print(json.dumps(json.loads(r.choices[0].message.content), indent=2))

{
  "company": "Amazon",
  "ticker": "AMZN",
  "quarter": "Q4",
  "revenue_b": 187.8,
  "revenue_growth_pct": 10,
  "aws_revenue_b": 24.2,
  "operating_income_b": 13.2,
  "guidance_low_b": 155,
  "guidance_high_b": 160
}


---
## 8. Batch Stock Ratings

In [79]:
stocks = {
    "Reliance (India)": "Conglomerate, P/E=28, Revenue Growth=+7.5%, Margin=8.2%, Jio 490M subs",
    "TCS (India)":      "IT Services, P/E=33, Revenue Growth=+4.2%, Margin=19%, 600K employees",
    "HDFC Bank (India)": "Banking, P/E=20, NII Growth=+10%, NPA=1.2%, Largest private bank",
    "NVDA (US)":        "Semiconductors, P/E=55, Revenue Growth=+94%, Margin=74%, AI leader",
    "TSLA (US)":        "EVs, P/E=170, Revenue Growth=+25%, Margin=18%, Beta=2.05",
    "MSFT (US)":        "Cloud/Software, P/E=36, Revenue Growth=+16%, Margin=70%, Azure+AI",
}

for name, data in stocks.items():
    r = client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[
            {"role": "system", "content": "In one sentence: Buy/Hold/Sell with the key reason."},
            {"role": "user", "content": f"{name}: {data}"}
        ],
        temperature=0.2, max_tokens=100,
    )
    print(f"{name}: {r.choices[0].message.content}\n")

Reliance (India): Buy: Reliance's diversified portfolio, strong revenue growth, and increasing subscriber base of Jio (490M) make it an attractive investment opportunity.

TCS (India): Buy: TCS has a strong market position, stable revenue growth, and a high margin, indicating a solid business model.

HDFC Bank (India): Buy: Strong growth prospects, high NII growth, and low NPA ratio indicate a stable and growing bank.

NVDA (US): Buy: Strong revenue growth, high margin, and leadership in AI make NVDA an attractive investment opportunity despite its high P/E ratio.

TSLA (US): Sell: The extremely high P/E ratio of 170 suggests that the stock is overvalued, despite strong revenue growth and high margins.

MSFT (US): Buy: Strong growth prospects in cloud and AI, high margin, and a dominant market position in software.



---
## 9. Beyond Finance — Fun Use Cases

In [80]:
# 9a. Cricket Match Analysis 🏏

match = """
Champions Trophy Final: India vs New Zealand, Dubai
India: 265/8 (50 ov) — Rohit 72(80), Kohli 56(65), Pandya 48*(32)
NZ Bowling: Santner 3/42, Boult 2/48, Ferguson 2/58
NZ: 241 all out (48.3 ov) — Williamson 89(102), Ravindra 43(51)
India Bowling: Bumrah 4/38, Kuldeep 2/45, Jadeja 2/41
India won by 24 runs. Bumrah: Player of the Match.
"""

r = client.chat.completions.create(
    model=SELECTED_MODEL,
    messages=[
        {"role": "system", "content": "You are Harsha Bhogle. Analyze with passion and insight. Keep it to 150 words."},
        {"role": "user", "content": f"Post-match analysis — turning point and key performances:\n{match}"}
    ],
    temperature=0.6, max_tokens=400,
)
print("🏏 CRICKET ANALYSIS\n")
print(r.choices[0].message.content)

🏏 CRICKET ANALYSIS

What a thrilling finale to the Champions Trophy. The turning point was undoubtedly Jasprit Bumrah's spell, where he picked up 4 crucial wickets for just 38 runs. His ability to contain and strike at the death was the difference between the two teams. Rohit Sharma's 72 and Virat Kohli's 56 set the tone for India, while Hardik Pandya's late flourish gave them a competitive total. For New Zealand, Kane Williamson's 89 was a lone beacon of hope, but Bumrah's brilliance proved too much to handle. A well-deserved victory for India, and a fitting recognition for Bumrah as the Player of the Match.


In [81]:
# 9b. Startup Pitch — Shark Tank India Style 🚀

r = client.chat.completions.create(
    model=SELECTED_MODEL,
    messages=[
        {"role": "system", "content": "You are a Shark Tank India judge. Sharp, actionable evaluation. Keep to 200 words."},
        {"role": "user", "content": """Evaluate my startup idea:
App connecting home cooks with people wanting ghar ka khana (like Uber for home food).
Target: Working professionals, PG students, elderly living alone.
Revenue: 15% commission + meal plan subscriptions.
I'm a Bangalore college student with ₹2L savings and basic coding skills.
What's good, what's missing, would you invest?"""}
    ],
    temperature=0.6, max_tokens=500,
)
print("🚀 STARTUP EVALUATION\n")
print(r.choices[0].message.content)

🚀 STARTUP EVALUATION

Interesting idea, but let's dive deeper. The concept has potential, targeting a specific demographic with a clear pain point. Your commission-based model and meal plan subscriptions are good revenue streams. 

However, I'm concerned about the competition, as similar platforms already exist. To stand out, you need a unique value proposition, such as stringent quality control, diverse cuisine options, or partnerships with local food suppliers.

As a college student with basic coding skills, I question your ability to scale the platform. You'll need a robust tech infrastructure and a team to manage operations, marketing, and customer support.

I'd invest ₹10L for 20% equity, but only if you can demonstrate a clear plan to address these concerns and showcase a functional MVP with a small, loyal customer base. Prove your execution capabilities, and we can discuss further.


In [82]:
# 9c. Exam Prep — Concepts Made Simple 📚

r = client.chat.completions.create(
    model=SELECTED_MODEL,
    messages=[
        {"role": "system", "content": "Expert teacher who explains using everyday Indian analogies. 3-4 sentences per concept max."},
        {"role": "user", "content": """Explain simply for a B.Tech/MBA student:
1. What is P/E ratio? (use chai stall analogy)
2. TCP vs UDP difference? (use real-life analogy)
3. What is gradient descent? (explain like ordering on Swiggy)"""}
    ],
    temperature=0.5, max_tokens=600,
)
print("📚 CONCEPTS MADE SIMPLE\n")
print(r.choices[0].message.content)

📚 CONCEPTS MADE SIMPLE

Here are the explanations:

1. The P/E ratio is like the price of a chai at a stall. Imagine you're buying a chai stall, and it makes Rs. 10 profit per year. If you buy the stall for Rs. 100, the P/E ratio is 10 (100/10), meaning you're paying Rs. 10 for every rupee of profit. A higher P/E ratio means you're paying more for each rupee of profit, just like overpaying for a chai.

2. TCP is like sending a registered letter, where the sender confirms receipt and ensures the message is delivered correctly. UDP is like sending a postcard, where you just send it and hope it reaches the recipient. In TCP, the sender waits for an acknowledgement before sending the next part of the message, ensuring reliability but slowing it down, whereas UDP is faster but may lose some data in transit.

3. Gradient descent is like ordering food on Swiggy. You start with an initial order (guess) and get a delivery (result) with a certain cost (error). Then, you adjust your order based o

---
## 10. Web-Grounded Analysis — DuckDuckGo + LLM

Search the web, then feed results to the LLM for grounded answers.

In [83]:
from ddgs import DDGS

def search_and_analyze(query, num_results=5):
    """Search DuckDuckGo, then ask the LLM to analyze results."""
    print(f'Searching: "{query}"\n')
    results = list(DDGS().text(query, max_results=num_results))
    if not results:
        print("No results found."); return

    context = ""
    for i, r in enumerate(results, 1):
        print(f"{i}. {r.get('title', '')}")
        print(f"   {r.get('body', '')[:100]}...\n")
        context += f"[{i}] {r.get('title','')}: {r.get('body','')}\n"

    print("=== AI Analysis ===\n")
    resp = client.chat.completions.create(
        model=SELECTED_MODEL,
        messages=[
            {"role": "system", "content": "Analyze search results to answer the question. Cite sources like [1], [2]. Be concise."},
            {"role": "user", "content": f"Question: {query}\n\nResults:\n{context}"}
        ],
        temperature=0.3, max_tokens=600,
    )
    print(resp.choices[0].message.content)

search_and_analyze("NVIDIA stock outlook 2026")

# Try more:
# search_and_analyze("Nifty 50 market outlook India 2026")
# search_and_analyze("IPL 2026 auction biggest buys")
# search_and_analyze("Best budget smartphones under 15000 India 2026")

Searching: "NVIDIA stock outlook 2026"

1. __symbol__ Stock Quote Price and Forecast | CNN
   14 hours ago -View NVIDIA Corporation NVDA stock quote prices, financial information, real-time fore...

2. 4 Reasons Why Nvidia Can Beat the Market Again in 2026 | The Motley Fool
   January 2, 2026 -Analysts expect revenue growth to slow to 50% in fiscal 2027, but this outlook coul...

3. NVIDIA (NVDA) Stock Forecast: Analyst Ratings, Predictions & Price Target 2026
   3 weeks ago -38 analysts have given NVIDIA (NVDA) aconsensus rating of Buywhile the NVIDIA (NVDA) pr...

4. NVIDIA (NVDA) Stock Forecast & Price Prediction 2026–2030 | CoinCodex
   3 weeks ago -In 2026, NVIDIA (NVDA) is anticipated to change hands in a trading channel between · $ ...

5. NVIDIA (NVDA) Stock Forecast and Price Target 2026 $NVDA
   5 hours ago -NVDA's current price target is $275.25.Learn why top analysts are making this stock for...

=== AI Analysis ===

Based on the search results, the NVIDIA stock outlook for

---

## 11. Multilingual — Hindi

Open-source models like `qwen/qwen3-32b` handle Hindi and other Indian languages well.

In [84]:
# 11a. Hindi — Mutual Funds explained in Hinglish
r = client.chat.completions.create(
    model="qwen/qwen3-32b",
    messages=[
        {"role": "system", "content": "You are a friendly college senior who explains finance in Hinglish (Hindi + English mix). Keep it casual and simple."},
        {"role": "user", "content": "Mujhe samjhao ki mutual funds kya hote hain aur SIP kaise start kare? Mere paas ₹500/month spare hai — kya ye enough hai?"}
    ],
    temperature=0.6, max_tokens=500,
)
print("🇮🇳 HINDI — Mutual Funds & SIP\n")
print(r.choices[0].message.content)

🇮🇳 HINDI — Mutual Funds & SIP

<think>
Okay, the user is asking about mutual funds and how to start an SIP with ₹500 a month. Let me break this down. First, I need to explain what mutual funds are in simple Hinglish. Maybe start by comparing them to a fund where many people pool their money. Then mention the different types like equity, debt, etc., but keep it brief.

Next, the SIP part. I should explain that SIP is a way to invest regularly, like a monthly installment. Emphasize the benefits like rupee cost averaging and discipline. The user has ₹500, so I need to confirm if that's enough. Most platforms do accept SIPs starting from ₹500, so that's good. Maybe mention some popular apps or platforms where they can start.

Also, the user might be a first-time investor, so I should highlight the importance of choosing the right fund based on their goals. Maybe suggest looking at expense ratio and past performance, but not get too technical. End with a positive note about starting early a

In [85]:
# 11b. Translation — English → Hindi (Devanagari)
text = "Artificial Intelligence is transforming how we work, learn, and communicate. India is emerging as a global AI hub."

r = client.chat.completions.create(
    model="qwen/qwen3-32b",
    messages=[
        {"role": "system", "content": "You are a professional translator. Translate accurately into Hindi using Devanagari script."},
        {"role": "user", "content": f"Translate this text into Hindi (Devanagari script).\n\nText: \"{text}\""}
    ],
    temperature=0.3, max_tokens=300,
)
print("🇮🇳 HINDI TRANSLATION\n")
print(f"English: {text}\n")
print(f"Hindi:   {r.choices[0].message.content}")

🇮🇳 HINDI TRANSLATION

English: Artificial Intelligence is transforming how we work, learn, and communicate. India is emerging as a global AI hub.

Hindi:   <think>
Okay, let me tackle this translation. The user wants the given English text translated into Hindi using Devanagari script. First, I need to make sure I understand the original text correctly. The sentence is: "Artificial Intelligence is transforming how we work, learn, and communicate. India is emerging as a a global AI hub."

Starting with the first part: "Artificial Intelligence is transforming how we work, learn, and communicate." 

"Artificial Intelligence" in Hindi is "कृत्रिम बुद्धिमत्ता" (Kritrim Buddhimatta). The verb "is transforming" would be "परिवर्तित कर रही है" (parivartit kar rahi hai). 

Now, "how we work, learn, and communicate" – "how we work" is "हम काम करते हैं" (ham kām karte hain), "learn" is "सीखते हैं" (sikhate hain), and "communicate" is "संचार करते हैं" (sanchār karte hain). So the phrase would be "ह

---
## 12. Multimodal — Image Understanding

`meta-llama/llama-4-scout-17b-16e-instruct` can analyze images. Pass any local image (base64 encoded).

In [55]:
# Analyze a local image (base64 encoded)
import base64

VISION_MODEL = "meta-llama/llama-4-scout-17b-16e-instruct"

def analyze_local_image(image_path, question="What's in this image? Describe it."):
    """Send a local image to Llama 4 Scout for analysis."""
    with open(image_path, "rb") as f:
        b64 = base64.b64encode(f.read()).decode("utf-8")
    ext = image_path.rsplit(".", 1)[-1].lower()
    mime = {"jpg": "image/jpeg", "jpeg": "image/jpeg", "png": "image/png", "gif": "image/gif", "webp": "image/webp"}.get(ext, "image/jpeg")
    r = client.chat.completions.create(
        model=VISION_MODEL,
        messages=[{"role": "user", "content": [
            {"type": "text", "text": question},
            {"type": "image_url", "image_url": {"url": f"data:{mime};base64,{b64}"}}
        ]}],
        temperature=0.5, max_tokens=500,
    )
    return r.choices[0].message.content

# Analyze the sample image included in the repo
result = analyze_local_image("sample_image.jpg", "Describe this image in detail. What landmark is this? What is its historical significance?")
print("🖼️ LOCAL IMAGE ANALYSIS\n")
print(result)

# Try with your own images:
# print(analyze_local_image("my_screenshot.png", "What does this screenshot show?"))
# print(analyze_local_image("my_chart.png", "Analyze this chart and identify the trend."))

🖼️ LOCAL IMAGE ANALYSIS

The image depicts the Taj Mahal, a majestic white marble mausoleum located in Agra, India. The landmark is characterized by its large central dome surrounded by four smaller domes and four minarets, one on each corner of the complex.

**Historical Significance:**

The Taj Mahal was built between 1632 and 1653 by Mughal Emperor Shah Jahan as a tribute to his beloved wife, Mumtaz Mahal, who died during the birth of their 14th child. The monument is considered one of the most beautiful examples of Mughal architecture, which blended Indian, Persian, and Islamic styles.

**Key Features:**

* **Architectural Style:** The Taj Mahal's design combines elements of Indian, Persian, and Islamic architectural styles, reflecting the cultural diversity of the Mughal Empire.
* **Symbolism:** The Taj Mahal symbolizes eternal love and is often referred to as a symbol of India's rich cultural heritage.
* **UNESCO World Heritage Site:** The Taj Mahal was designated a UNESCO World 

---
## Summary

| # | Section | Model Used | Highlights |
|---|---------|------------|------------|
| 1-2 | Setup & Models | — | 12+ free models from Meta, OpenAI, Alibaba, Moonshot |
| 3 | Basic Chat | `llama-3.3-70b-versatile` | OpenAI-compatible API basics |
| 4 | Sentiment | `llama-3.3-70b-versatile` | NVIDIA headline scoring |
| 5 | Investment Thesis | `llama-3.3-70b-versatile` | Reliance Industries analysis |
| 6 | Model Comparison | 6 models | Same prompt — compare speed & quality |
| 7 | Streaming + JSON | `llama-3.3-70b-versatile` | Real-time output & structured extraction |
| 8 | Batch Ratings | `llama-3.1-8b-instant` | 6 stocks rated in seconds |
| 9 | Fun Use Cases | `llama-3.3-70b-versatile` | Cricket, startup pitch, exam prep |
| 10 | Web Search + LLM | `llama-3.3-70b-versatile` | DuckDuckGo grounded analysis |
| 11 | Multilingual | `qwen/qwen3-32b` | Hindi (Hinglish) + Hindi translation |
| 12 | Multimodal | `llama-4-scout-17b` | Local image analysis |

### Try Experimenting!
1. Change `SELECTED_MODEL` and re-run — compare quality across models
2. Adjust `temperature` (0.0 = factual, 1.0 = creative)
3. Replace stock data with your favorite companies
4. Try `qwen/qwen3-32b` for Hindi and other languages
5. Try `meta-llama/llama-4-scout-17b-16e-instruct` with your own images
6. Modify the DuckDuckGo queries to search anything in real-time